<a href="https://colab.research.google.com/github/ElMartinez31/Data_Science/blob/main/Udacity_PEFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Apply lightweight finetuning to a fundation Model

# Tasks to perform
# 1. Load a pre-trained model and evaluate its performance
# 2. Perform parameter-efficient fine tuning using the pre-trained model
# 3. Perform inference using the fine-tuned model and compare its performance to the original model

In [ ]:
# Training a model using Hugging Face PEFT requires two additional steps beyond traditional fine-tuning:

# Creating a PEFT config
# Converting the model into a PEFT model using the PEFT config
# Inference using a PEFT model is almost identical to inference using a non-PEFT model. The only difference is that it must be loaded as a PEFT model.

In [ ]:
import json, pathlib

nb_path = "Udacity_PEFT.ipynb"  # adapte le chemin
p = pathlib.Path(nb_path)
nb = json.loads(p.read_text(encoding="utf-8"))

# retirer le bloc fautif
nb.get("metadata", {}).pop("widgets", None)

# (facultatif) nettoyer les sorties pour alléger
for cell in nb.get("cells", []):
    if cell.get("cell_type") == "code":
        cell["outputs"] = []
        cell["execution_count"] = None

p.write_text(json.dumps(nb, ensure_ascii=False, indent=1), encoding="utf-8")
print("Notebook nettoyé. Re-commit/push.")


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install -U transformers datasets peft accelerate evaluate


In [ ]:
# load dataset

from datasets import load_dataset

ds = load_dataset("prasadsawant7/sentiment_analysis_preprocessed_dataset")
ds

In [ ]:
train_df = ds["train"]
train_df

test_df = ds["test"]
test_df[0] # exaple

In [ ]:
# load tokenizer

from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained("gpt2")

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

tokenizer.padding_side = "left"  # GPT-2 en classification padding à gauche



In [ ]:


tok = tokenizer(["Hommage à ce brave Jean Pormanove","Justice pour JP"], truncation=True)


# print("inputs id: \n", tok["input_ids"])

for id in tok["input_ids"]:
  print("id: ", id)
  print("\n", tokenizer.convert_ids_to_tokens(id))





In [ ]:
# here we track the missing text of train
bad_none = ds["train"].filter(lambda x: x["text"] is None)
bad_none["text"]



In [ ]:
# function to replace None text to " "
def sanitize_data(batch):
  texts = batch["text"]
  texts = [" " if (t is None) else t for t in texts]
  return {"text" : texts}

ds = ds.map(sanitize_data, batched = True) # apply


In [ ]:
# now lets apply the tokenizer to our dataset

def tokenize_sentences(batch):
  token = tokenizer(batch["text"], truncation = True, max_length=128) # no padding here
  return token

ds_token_train = ds["train"].map(tokenize_sentences, batched = True, batch_size = 1000)
ds_token_train

ds_token_test = ds["test"].map(tokenize_sentences, batched = True, batch_size = 1000)
# had to sanitize the data as we got an error due to None text


# advantages of working with batch:
# one call of a function with 1000 snetences is performing far better than 1000 calls of 1 sentence.
#

In [ ]:
# 1st example
ds_token_train[0] # we have to tokens id and attention masks

In [ ]:
ds_token_train

import torch
from torch.utils.data import TensorDataset, DataLoader

# we can now only keep labels, input_ids et attention_mask

columns_to_keep = ["labels", "input_ids", "attention_mask"]

ds_token_train_ready = ds_token_train.select_columns(columns_to_keep) # format ok for HF
ds_token_test_ready = ds_token_test.select_columns(columns_to_keep)

ds_token_train_torch = ds_token_train_ready.set_format(type="torch", columns=columns_to_keep) # format for pytorch loop training
ds_token_test_torch = ds_token_test_ready.set_format(type="torch", columns=columns_to_keep)


# collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# convert to dataloader
train_loader = DataLoader(ds_token_train_ready, batch_size=16, shuffle=True, collate_fn=data_collator)
test_loader  = DataLoader(ds_token_test_ready,  batch_size=16, shuffle = False, collate_fn=data_collator)

In [ ]:
# we have the right columns

In [ ]:
# %pip install -q peft

from peft import LoraConfig, get_peft_model, PeftModel, TaskType

In [ ]:
# load model

from transformers import AutoModelForSequenceClassification

# load and provide params
model = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=3,
    id2label={0: "BAD", 1: "NEU", 2:"GOOD"},
    label2id={"BAD": 0, "NEU": 1, "GOOD" : 2},
)

# doing so gives us a final output layer of 3 neurons, meaning that we will need to use crossentropy as loss function


# Très important: agrandir les embeddings pour le nouveau vocab
model.resize_token_embeddings(len(tokenizer)) # car on a jouté un pad token
# => sinon erreur index out of range
# Indiquer au modèle quel est le pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id

# === config LoRA ===
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                # rang LoRA (8 assez bon compromis)
    lora_alpha=16,      # échelle
    lora_dropout=0.1,   # dropout LoRA, appliqué uniquement au chemin LoRA pendant l’entraînement
    # Cibles typiques pour GPT-2 (attention + MLP):
    target_modules=["c_attn", "c_fc", "c_proj"], #projection QKV de l’attention, projection montante du MLP, projection de sortie (attention/MLP selon le bloc)
    bias="none",
    modules_to_save=["score"]   # <<< garder la tête trainable
)

# performance:
#Si sous-apprentissage → augmente r (16/32) ou alpha, ou cible plus de modules.
#Si surapprentissage → monte un peu lora_dropout (0.2), baisse alpha, ou gèle davantage en dehors de LoRA.

# on enveloppe le modèle avec LoRA
model = get_peft_model(model, peft_config) # seuls les parametres LoRA ne sont pas gelés

# contrôle que seules les couches LoRA sont entraînables
model.print_trainable_parameters()



In [ ]:
# 2 ways to do the training 1/ manually with pytorch 2/ automate with a trainer HF



In [ ]:
# 1/ with pytorch:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import  get_linear_schedule_with_warmup, DataCollatorWithPadding

In [ ]:


optim = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
# on filtre sur les parametres non gelés (fait au moment du wrapping LoRA)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

# criterion = nn.CrossEntropyLoss()# no need here because the automodel already picks automatically the right loss according to the outputs number


# Train
epochs = 2

for epoch in range(epochs):
  model.train()
  steps, running_loss = 0, 0.0
  for step, batch in enumerate(train_loader, start = 1): # 1 step = 1 batch within the epoch
    optim.zero_grad()
    batch = {k:v.to(device) for k, v in batch.items()} # # dict: input_ids, attention_mask, labels = our batch
    outputs = model(**batch) # loss included in the model if we provide the labels parameter (here we do)
    loss = outputs.loss
    loss.backward() # backprop
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # always after the loss.backward, used to avoid exploding gradients
    optim.step() # adjust weights
    running_loss += loss.item()
    if step % 100 == 0 or step == len(train_loader):
      print(f'step : {step} over {len(train_loader)} for epoch : {epoch}, currnet loss: {running_loss}, avg_loss : {running_loss / step}')





In [ ]:
# save model weights + tokenizer
adapter_dir = "/content/drive/Othercomputers/Mon ordinateur portable/Data Science/Datasets/gpt2_lora_adapters"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)


# load saved model
from peft import PeftModel

base = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=3,
    id2label={0:"BAD",1:"NEU",2:"GOOD"},
    label2id={"BAD":0,"NEU":1,"GOOD":2},
)

base.resize_token_embeddings(len(tokenizer))
base.config.pad_token_id = tokenizer.pad_token_id
base.to(device)

model = PeftModel.from_pretrained(base, adapter_dir).to(device)


In [ ]:
# eval

import torch


@torch.no_grad()
def eval_loop(model, loader):
  correct = 0
  correct_total = 0
  total = 0
  running_loss = 0.0
  steps = 0

  for batch in loader:
          batch = {k: v.to(device) for k, v in batch.items()}   # input_ids, attention_mask, labels

          outputs = model(**batch) # loss dispo si 'labels' est présent
          loss = outputs.loss
          logits = outputs.logits #ici on a besoin des logits pour comparer les prédictions aux vrais labels
          preds = logits.argmax(dim=-1) # prédictions

          running_loss += loss.item()
          steps += 1

          labels = batch["labels"] # vrais labels sur le batch
          correct += (preds == labels).sum().item() # corrects += corrects du batch
          total += labels.size(0) # total += nb d'exemples du batch

  avg_loss = running_loss / max(steps, 1)
  accuracy = correct / max(total, 1)
  return {"accuracy" : accuracy,
          "average_loss": avg_loss}


res = eval_loop(model = model, loader =test_loader)

print("LoRA model accuracy : ", res["accuracy"])



In [ ]:
# # autre metriques a considérer

# import numpy as np
# all_preds, all_labels = [], []

# with torch.no_grad():
#     for batch in test_loader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         logits = model(**batch).logits
#         preds = logits.argmax(dim=-1)

#         all_preds.append(preds.cpu().numpy())
#         all_labels.append(batch["labels"].cpu().numpy())

# all_preds = np.concatenate(all_preds)
# all_labels = np.concatenate(all_labels)

# acc = (all_preds == all_labels).mean()
# print(f"accuracy: {acc:.4f}  ({(all_preds==all_labels).sum()}/{len(all_labels)})")

# # (optionnel) F1 macro si scikit-learn dispo
# # from sklearn.metrics import f1_score, classification_report
# # print("F1 macro:", f1_score(all_labels, all_preds, average="macro"))
# # print(classification_report(all_labels, all_preds, target_names=["BAD","NEU","GOOD"])

In [ ]:
# now let's eval our task on the fresh base model


base = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=3,
    id2label={0:"BAD",1:"NEU",2:"GOOD"},
    label2id={"BAD":0,"NEU":1,"GOOD":2},
)

base.resize_token_embeddings(len(tokenizer))
base.config.pad_token_id = tokenizer.pad_token_id
base.to(device)


base_res = eval_loop(base, test_loader)

print("base model accuracy: ", base_res["accuracy"])

In [ ]:
# # to do

# # export to github


# # perform the right project structure (model, resutls,readme, ...)

# peft-sentiment/
# ├─ README.md
# ├─ requirements.txt
# ├─ notebooks/
# │  └─ peft_sentiment.ipynb
# ├─ models/
# │  └─ gpt2_lora_adapters/        # adapters LoRA sauvegardés ici
# ├─ results/
# │  ├─ baseline_metrics.json
# │  └─ lora_metrics.json
# ├─ .gitignore
# └─ .gitattributes   # (si Git LFS nécessaire)


# # requirements:
# torch>=2.1
# transformers>=4.40
# peft>=0.10
# datasets>=2.18
# scikit-learn>=1.3

# # gitignore
# __pycache__/
# .ipynb_checkpoints/
# *.pyc
# .env
# .venv
# data/
# cache/

# # save in relative path more than on drive (drop drive)
# from pathlib import Path
# save_dir = Path("models/gpt2_lora_adapters")
# save_dir.mkdir(parents=True, exist_ok=True)
# model.save_pretrained(save_dir.as_posix())
# tokenizer.save_pretrained(save_dir.as_posix())

# #  clean the code (provide clean part for results comparison, libraries loading ...)
# # Baseline
# baseline_metrics = eval_loop(baseline_model, test_loader)
# print("Baseline:", baseline_metrics)

# # LoRA
# lora_metrics = eval_loop(lora_model, test_loader)
# print("LoRA:", lora_metrics)

# # Tableau récap
# import pandas as pd
# pd.DataFrame([baseline_metrics, lora_metrics], index=["baseline","lora"])

# # les sauver dans les fichiers adaptés
# import os, json
# os.makedirs("results", exist_ok=True)
# with open("results/baseline_metrics.json","w") as f: json.dump(baseline_metrics, f, indent=2) # cree un f pour les res du baseline model
# with open("results/lora_metrics.json","w") as f: json.dump(lora_metrics, f, indent=2) # crée un fichier pour les resultats du lora model

# # Provide the github link for the project